# 13. ord() and chr(): characters and code points

A Unicode code point is an integer assigned to a character. Python strings store Unicode text: `ord()` and `chr()` let you move between a **one-character string** and its **integer code point**.

| Function | Input | Output | Example |
|---|---|---|---|
| `ord(char)` | One-character string | Integer code point | `ord('h') == 104` |
| `chr(number)` | Integer code point | One-character string | `chr(104) == 'h'` |

We will follow the encoder and decoder exercise in your screenshots: add a key to encode, subtract the same key to decode. Run the cells from top to bottom in Colab (**Runtime → Run all**). No installation is needed.


## 1. A character and its number are different values

`'h'` is a string; `104` is an integer. `ord('h')` returns `104`, and `chr(104)` returns `'h'`. These functions undo each other: `chr(ord(char)) == char`.

Unicode includes more than English letters. Also, `ord('7')` is the code point of the digit character, **not** the number 7. Use `int('7')` to read a number from text.


In [1]:
char = 'h'
code_point = ord(char)
print(repr(char), type(char).__name__, '->', code_point, type(code_point).__name__)
print(code_point, '->', repr(chr(code_point)))

for char in ['A', '7', '中']:
    print(repr(char), '->', ord(char), '->', repr(chr(ord(char))))
    assert chr(ord(char)) == char

print("ord('7'):", ord('7'))  # 55: a Unicode code point
print("int('7'):", int('7'))  # 7: a numeric value


'h' str -> 104 int
104 -> 'h'
'A' -> 65 -> 'A'
'7' -> 55 -> '7'
'中' -> 20013 -> '中'
ord('7'): 55
int('7'): 7


## 2. Your encoder: add the key

The screenshot asks for `encoder(char, key=200)`. Trace `encoder('h')`:

| Step | Expression | Value | Type |
|---|---|---|---|
| Start | `char` | `'h'` | `str` |
| Get its code point | `ord(char)` | `104` | `int` |
| Add the default key | `104 + 200` | `304` | `int` |
| Convert back | `chr(304)` | `'İ'` | `str` |

A call to `ord(char)` returns a value; it does not change `char`. Store or use that return value. Arithmetic happens on the integer, so `char + key` would be a type error.

The expanded function makes each step visible. The one-line version matches your screenshot and evaluates from the inner call outward.


In [2]:
def encoder(char, key=200):
    code_point = ord(char)
    shifted_code_point = code_point + key
    encoded_char = chr(shifted_code_point)
    return encoded_char

def encoder_short(char, key=200):
    return chr(key + ord(char))

original = 'h'
encoded = encoder(original)
print('Original:', repr(original))
print('Code point:', ord(original))
print('After adding 200:', ord(original) + 200)
print('Encoded:', repr(encoded))
assert encoded == 'İ'
assert encoder_short(original) == encoded

# Reproduce the five character checks from the exercise.
for char, expected in zip('hello', 'İĭĴĴķ'):
    assert encoder(char) == expected
print('All five encoder checks passed.')


Original: 'h'
Code point: 104
After adding 200: 304
Encoded: 'İ'
All five encoder checks passed.


## 3. Your decoder: subtract the same key

Encoding moved `104 → 304`; decoding moves `304 → 104`.

`decoder('İ')` evaluates `ord('İ')` to `304`, subtracts `200`, then returns `chr(104)`, which is `'h'`. Using the same key gives `decoder(encoder(char, key), key) == char`, provided the shifted code point stays in range.

Each function handles **one character**. To process `'hello'`, call it for each character and use `''.join(...)` to combine the results. Omitting the key uses 200; passing 500 overrides the default in both directions.


In [3]:
def decoder(encoded_char, key=200):
    char = chr(ord(encoded_char) - key)
    return char

print("'İ' ->", ord('İ'), '->', ord('İ') - 200, '->', repr(decoder('İ')))
for encoded_char, expected in zip('İĭĴĴķ', 'hello'):
    assert decoder(encoded_char) == expected

message = 'hello'
encoded_message = ''.join(encoder(char) for char in message)
decoded_message = ''.join(decoder(char) for char in encoded_message)
print('Default key 200:', repr(message), '->', repr(encoded_message), '->', repr(decoded_message))
assert decoded_message == message

key = 500
encoded_with_500 = ''.join(encoder(char, key) for char in message)
decoded_with_500 = ''.join(decoder(char, key) for char in encoded_with_500)
print('Custom key 500:', repr(message), '->', repr(encoded_with_500), '->', repr(decoded_with_500))
assert decoded_with_500 == message
print('All decoder and round-trip checks passed.')


'İ' -> 304 -> 104 -> 'h'
Default key 200: 'hello' -> 'İĭĴĴķ' -> 'hello'
Custom key 500: 'hello' -> 'ɜəɠɠɣ' -> 'hello'
All decoder and round-trip checks passed.


## 4. Input rules and common mistakes

For text, `ord()` requires exactly one Unicode code point: `''` and `'hello'` are invalid. Some visible symbols contain several code points, so one visible symbol is not always a valid input.

`chr()` accepts integer code points from `0` through `0x10FFFF` (1,114,111). A shift outside that range raises `ValueError`; this exercise does not wrap around automatically. Some accepted code points are nonprinting or surrogates, so an accepted number does not always produce a displayable or UTF-8-encodable character.

The errors below are caught so **Run all** can finish.


In [4]:
examples = [
    ("ord('')", lambda: ord('')),
    ("ord('hello')", lambda: ord('hello')),
    ("'h' + 200", lambda: 'h' + 200),
    ("chr('104')", lambda: chr('104')),
    ("chr(-1)", lambda: chr(-1)),
    ("chr(0x110000)", lambda: chr(0x110000)),
]
for label, operation in examples:
    try:
        operation()
    except (TypeError, ValueError) as error:
        print(label, '->', type(error).__name__)

assert encoder('h', 0) == 'h'
assert decoder(encoder('h', 500), 500) == 'h'
assert decoder(encoder('h', 500), 200) != 'h'
print('A zero key changes nothing; decoding needs the matching key.')


ord('') -> TypeError
ord('hello') -> TypeError
'h' + 200 -> TypeError
chr('104') -> TypeError
chr(-1) -> ValueError
chr(0x110000) -> ValueError
A zero key changes nothing; decoding needs the matching key.


**Remember:** `ord` gives the number; `chr` gives the character. Encode with `chr(ord(char) + key)`; decode with `chr(ord(encoded_char) - key)`.

This is a code point shifting exercise, not secure encryption or a conversion to UTF-8 bytes. The function name `encoder` here is defined by the exercise.

References: [Python ord()](https://docs.python.org/3/library/functions.html#ord) · [Python chr()](https://docs.python.org/3/library/functions.html#chr).
